In [2]:
import os

# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import BaseChronosPipeline, Chronos2Pipeline
import holidays
from pandas.tseries.frequencies import to_offset


# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

In [3]:
def regularize_hourly(g: pd.DataFrame) -> pd.DataFrame:
    """
    Reindex each group's timestamps to strict hourly and fill gaps.
    Works whether the grouping column is present or omitted (include_groups=False).
    """
    # The group key (id) is available as g.name; if ID_COL exists, prefer it.
    sid = g[ID_COL].iloc[0] if ID_COL in g.columns else g.name

    g = g.sort_values(TS_COL)
    full_idx = pd.date_range(g[TS_COL].min(), g[TS_COL].max(), freq="h")
    g = g.set_index(TS_COL).reindex(full_idx)
    g.index.name = TS_COL

    # restore id (constant for the whole group)
    g[ID_COL] = sid

    # numeric + fill for targets
    for col in TARGETS:
        if col in g.columns:
            g[col] = pd.to_numeric(g[col], errors="coerce").ffill().bfill()
    return g.reset_index()

def add_holiday_flags(
    df: pd.DataFrame,
    ts_col: str = "ds",
    local_tz: str = "America/Montreal",
    observed: bool = True,
    include_names: bool = False,
) -> pd.DataFrame:
    """
    Adds boolean columns:
      • is_qc_holiday       — Québec public holiday (CA-QC)
      • is_jewish_holiday   — Israeli public/Jewish holiday (IL)
    Optionally adds:
      • qc_holiday_name
      • jewish_holiday_name

    Notes:
      • Holiday checks are date-based (00:00–24:00 local calendar date),
        not sundown-to-sundown observance.
      • NaT timestamps are ignored gracefully.
    """
    out = df.copy()

    # 1) Parse to datetime
    out[ts_col] = pd.to_datetime(out[ts_col], errors="coerce")

    # 2) Get the calendar DATE to use for holiday lookup
    #    - If tz-aware: convert to Montreal then take .date
    #    - If naive: assume values already represent local Montreal wall-clock; just take .date
    if getattr(out[ts_col].dt, "tz", None) is not None:
        dates_for_calendar = out[ts_col].dt.tz_convert(local_tz).dt.date
    else:
        dates_for_calendar = out[ts_col].dt.date

    # 3) Build a SAFE integer year range for the holiday objects
    years_series = pd.Series(dates_for_calendar)
    years_series = years_series.dropna().map(lambda d: int(pd.Timestamp(d).year))
    if years_series.empty:
        raise ValueError("No valid datetimes found to extract holiday years.")
    years = list(range(int(years_series.min()), int(years_series.max()) + 1))

    # 4) Construct holiday calendars
    qc_holidays = holidays.Canada(subdiv="QC", years=years, observed=observed)
    il_holidays = holidays.Israel(years=years, observed=observed)

   # 5) Flag membership
    out["is_qc_holiday"] = [ ("yes" if d in qc_holidays else "no") if pd.notna(pd.Timestamp(d)) else "no"
                             for d in dates_for_calendar ]
    out["is_jewish_holiday"] = [ ("yes" if d in il_holidays else "no") if pd.notna(pd.Timestamp(d)) else "no"
                                 for d in dates_for_calendar ]

    if include_names:
        out["qc_holiday_name"] = [ qc_holidays.get(d, "no") if pd.notna(pd.Timestamp(d)) else "no"
                                   for d in dates_for_calendar ]
        out["jewish_holiday_name"] = [ il_holidays.get(d, "no") if pd.notna(pd.Timestamp(d)) else "no"
                                       for d in dates_for_calendar ]

    return out

shift_types_dict = {'W1':'flow',
 'X1':'pod',
 'X3':'pod',
 'X4':'vertical',
 'X2':'vertical',
 'WOC1':'oncall',
 'WOC2':'oncall',
 'WOC3':'oncall',
 'X5':'pod',
 'W3':'overlap',
 'Y1':'pod',
 'Y3':'pod',
 'Y4':'vertical',
 'Y2':'vertical',
 'Y5':'pod',
 'Z1':'night',
 'Z2':'night',
 'D1':'pod',
 'R1':'pod',
 'P1':'vertical',
 'D2':'vertical',
 'OC1':'oncall',
 'OC2':'oncall',
 'V1':'flow',
 'A1':'pod',
 'G1':'vertical',
 'E1':'pod',
 'R2':'pod',
 'A2':'pod',
 'P2':'vertical',
 'E2':'vertical',
 'N1':'night',
 'N2':'night',
 'L2':'overlap',
 'L4':'overlap',
 'H1':'teaching',
 'B1':'vertical',
 'L1':'overlap',
 'W5':'overlap',
 'L6':'overlap',
 'B2':'vertical'}

In [4]:
# Load hourly data
df = pd.read_csv(
    'https://www.dropbox.com/scl/fi/s83jig4zews1xz7vhezui/allDataWithCalculatedColumns.csv?rlkey=9mm4zwaugxyj2r4ooyd39y4nl&raw=1')
df.ds = pd.to_datetime(df.ds, errors="coerce")
df['id'] = 'jgh'

# Load shift data
all_shifts_df = pd.read_csv('https://www.dropbox.com/scl/fi/yeyr2a7pj6nry8i2q3m0c/all_shifts.csv?rlkey=q1su2h8fqxfnlu7t1l2qe1w0q&raw=1')
all_shifts_df['shift_start'] = pd.to_datetime(all_shifts_df['shift_start']).dt.round('h')
all_shifts_df['shift_end'] = pd.to_datetime(all_shifts_df['shift_end']).dt.round('h')
all_shifts_df['shift_type'] = all_shifts_df['shift_short_name'].map(shift_types_dict)

# Create hourly rows
# We'll use a list comprehension to generate the range for each row
expanded_rows = []
for _, row in all_shifts_df.iterrows():
    # Create range. inclusive='left' means [start, end)
    # If start == end (e.g. 0 length shift after rounding), it will be empty, which is correct
    hours = pd.date_range(row['shift_start'], row['shift_end'], freq='h', inclusive='left')
    for h in hours:
        expanded_rows.append({
            'ds': h,
            'user': row['first_name']+row['last_name'],
            'shift_type': row['shift_type'],
            'shift_short_name': row['shift_short_name']
        })

expanded_df = pd.DataFrame(expanded_rows)

# Pivot
# index=timestamp, columns=user_id, values=shift_type
hourly_shifts_by_user_df = expanded_df.pivot_table(
    index='ds', 
    columns='user', 
    values='shift_type', 
    aggfunc='first' # In case of duplicates, take the first
)

# Fill NaNs
hourly_shifts_by_user_df = hourly_shifts_by_user_df.fillna('NotWorking')



ID_COL = "id"
TS_COL = "ds"
# TARGETS = ['total_tbs', 'Inflow_Total', 'overflow']
# Targets are all columns in df except ds (timestamp) and id
# TARGETS = [col for col in df.columns.tolist() if col != TS_COL and col != ID_COL]
TARGETS = ['TTStr']

df = df.copy()
df[TS_COL] = pd.to_datetime(df[TS_COL], errors="coerce")
df = df.dropna(subset=[TS_COL])

# Snap to exact hours (lowercase 'h' to avoid FutureWarning)
df[TS_COL] = df[TS_COL].dt.floor("h")

# Sort + dedupe
df = df.sort_values([ID_COL, TS_COL]).drop_duplicates(
    [ID_COL, TS_COL], keep="last")





# Call apply with include_groups=False if supported; else fall back
gb = df.groupby(ID_COL, group_keys=False)
try:
    df = gb.apply(regularize_hourly, include_groups=False)
except TypeError:
    # older pandas without include_groups
    df = gb.apply(regularize_hourly)

# Assert truly hourly (accept 'h' and 'H')
g = df[df[ID_COL] == "jgh"].sort_values(TS_COL)
freq = pd.infer_freq(g[TS_COL])
if not freq:
    raise ValueError("No inferable frequency after regularization.")
if to_offset(freq).name.lower() != "h":
    # extra check independent of infer_freq
    diffs = g[TS_COL].diff().dropna()
    bad = g.loc[diffs != pd.Timedelta(hours=1), TS_COL].head(10).tolist()
    raise ValueError(f"Non-1h gaps remain around: {bad}")

In [5]:
from tqdm.notebook import tqdm
import pandas as pd


weather_df = pd.read_csv('https://www.dropbox.com/scl/fi/gmhwwld9z9yychg4r0yuk/weather.csv?rlkey=66c78m90aviamr0x0uu72pfr8&raw=1')
weather_df.ds = pd.to_datetime(weather_df.ds, errors="coerce")

df_with_staffing = df.merge(hourly_shifts_by_user_df, on='ds')
all_variable_df = add_holiday_flags(df_with_staffing, ts_col='ds', include_names=True).merge(weather_df, on='ds')

# Define the start and end dates for November 2025
start_date = pd.Timestamp('2025-11-01 00:00:00')
end_date = pd.Timestamp('2025-11-30 23:00:00')
# end_date = pd.Timestamp('2025-11-01 3:00:00')

output_df = pd.DataFrame()


# Generate a date range for each hour in November 2025
hourly_timestamps = pd.date_range(start=start_date, end=end_date, freq='h')

# Iterate through the timestamps with a progress bar
for timestamp in tqdm(hourly_timestamps, desc="Processing November 2025 hours"):
  target_df = df[df['ds'] < timestamp]
  # print(target_df.tail(5))
  # df_with_holidays = add_holiday_flags(target_df, ts_col='ds', include_names=True)

  # #create a dataframe with the next 24 hours timestamps hourly as column 'ds', with column 'id' jgh
  # future_df = hourly_shifts_by_user_df.reset_index()[hourly_shifts_by_user_df.reset_index()['ds'] > target_df['ds'].max()].head(24)
  # future_df['id'] = 'jgh'
  # future_df = add_holiday_flags(future_df, ts_col='ds', include_names=True)

  # # First, add holiday flags to future_df
  # future_df_with_added_holidays = add_holiday_flags(future_df, ts_col='ds', include_names=True)

  # # Then, select only the columns from future_df_with_added_holidays that are also in df_with_holidays
  # common_columns = [col for col in future_df_with_added_holidays.columns if col in df_with_holidays.columns]
  # future_df_with_holidays = future_df_with_added_holidays[common_columns]

  future_df_with_staffing = hourly_shifts_by_user_df.reset_index()[hourly_shifts_by_user_df.reset_index()['ds'] > target_df['ds'].max()].head(24)
  future_df_with_staffing['id'] = 'jgh'

  future_weather_df = weather_df[weather_df.ds > target_df.ds.max()].head(24)
  future_weather_df['id']='jgh'


  forecast_all_vars_with_future = pipeline.predict_df(
      all_variable_df[all_variable_df.ds<timestamp],
      prediction_length=24,
      #future_df should be future_df_with_staffing merged with future_weather_df on 'ds' and 'id'
      future_df = future_df_with_staffing.merge(future_weather_df, on=['ds', 'id']),
      # quantile_levels=[0.2, 0.5, 0.8],
      quantile_levels=[0.5],
      id_column=ID_COL,
      timestamp_column=TS_COL,
      target=TARGETS,
    )
  forecast_all_vars_with_future['cutoff_ts'] = timestamp
  # add the real value from df at each value of ds by merging the dataframes on ds, keeping only the TTStr column from df
  forecast_all_vars_with_future = forecast_all_vars_with_future.merge(df[['ds', 'TTStr']], on='ds', how='left')
  # print(forecast_all_vars_with_future
  # print(forecast_all_vars_with_future[['cutoff_ts', 'ds', 'TTStr', 'predictions']])
  output_df = pd.concat([output_df, forecast_all_vars_with_future[['cutoff_ts', 'ds', 'TTStr', 'predictions']]])

output_df





/tmp/ipykernel_1072612/586953141.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_timestamps = pd.date_range(start=start_date, end=end_date, freq='H')


Processing November 2025 hours:   0%|          | 0/720 [00:00<?, ?it/s]

,cutoff_ts,ds,TTStr,predictions
0,2025-11-01 00:00:00,2025-11-01 00:00:00,90.0,88.258255
1,2025-11-01 00:00:00,2025-11-01 01:00:00,85.0,85.618912
2,2025-11-01 00:00:00,2025-11-01 02:00:00,82.0,83.982170
3,2025-11-01 00:00:00,2025-11-01 03:00:00,81.0,82.969688
4,2025-11-01 00:00:00,2025-11-01 04:00:00,80.0,82.841812
...,...,...,...,...
19,2025-11-30 23:00:00,2025-12-01 18:00:00,130.0,110.570389
20,2025-11-30 23:00:00,2025-12-01 19:00:00,110.0,106.360626
21,2025-11-30 23:00:00,2025-12-01 20:00:00,109.0,103.306206
22,2025-11-30 23:00:00,2025-12-01 21:00:00,104.0,98.320892


In [6]:
output_df.to_csv('nov2025_TTStr_forecasts.csv', index=False)

In [7]:
output_backup = output_df.copy()

In [8]:
forecast = output_df.copy()

In [10]:
#calculate rmse and mape of the forecasts
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
# drop rows with NaN in TTStr
eval_df = forecast.dropna(subset=['TTStr'])
rmse = root_mean_squared_error(eval_df['TTStr'], eval_df['predictions'])
mape = mean_absolute_percentage_error(eval_df['TTStr'], eval_df['predictions'])
print(f"RMSE: {rmse}")
print(f"MAPE: {mape}")

RMSE: 10.536645711404264
MAPE: 0.09201491406456633


In [19]:
#calculate rmse and mape at each forecast horizon (1h ahead, 2h ahead, etc.)
eval_df['horizon'] = (eval_df['ds'] - eval_df['cutoff_ts']).dt.total_seconds() / 3600
horizon_metrics = eval_df.groupby('horizon').apply(lambda x: pd.Series({
    'rmse': root_mean_squared_error(x['TTStr'], x['predictions']),
    'mape': mean_absolute_percentage_error(x['TTStr'], x['predictions'])
}))

horizon_metrics.reset_index().to_csv('nov2025_TTStr_horizon_metrics.csv', index=False)

horizon_metrics

/tmp/ipykernel_1072612/2614563301.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  horizon_metrics = eval_df.groupby('horizon').apply(lambda x: pd.Series({


,rmse,mape
horizon,,
0.0,4.896081,0.041320
1.0,6.477773,0.055311
2.0,7.668936,0.066002
3.0,8.446093,0.072767
4.0,9.045922,0.077845
5.0,9.603523,0.083484
6.0,9.982663,0.088383
7.0,10.244334,0.091137
8.0,10.532342,0.094072


In [13]:
import dropbox
from dotenv import load_dotenv
import requests
from utils import upload
import os

load_dotenv()

dropbox_app_key = os.environ.get("DROPBOX_APP_KEY")
dropbox_app_secret = os.environ.get("DROPBOX_APP_SECRET")
dropbox_refresh_token = os.environ.get("DROPBOX_REFRESH_TOKEN")

token_url = "https://api.dropboxapi.com/oauth2/token"
params = {
    "grant_type": "refresh_token",
    "refresh_token": dropbox_refresh_token,
    "client_id": dropbox_app_key,
    "client_secret": dropbox_app_secret
}
r = requests.post(token_url, data=params)

dropbox_access_token = r.json()['access_token']

dbx = dropbox.Dropbox(dropbox_access_token)




In [15]:
upload(dbx, 'nov2025_TTStr_forecasts.csv', '', '',
            'nov2025_TTStr_forecasts.csv', overwrite=True)

uploaded as b'nov2025_TTStr_forecasts.csv'


FileMetadata(client_modified=datetime.datetime(2026, 3, 31, 16, 25, 40), content_hash='e770b9804b46e6383d950eb5f4d80e65940e18b602942017124f878edda903f8', export_info=NOT_SET, file_lock_info=NOT_SET, has_explicit_shared_members=NOT_SET, id='id:oNSmVCFixyQAAAAAAAB84w', is_downloadable=True, media_info=NOT_SET, name='nov2025_TTStr_forecasts.csv', parent_shared_folder_id=NOT_SET, path_display='/nov2025_TTStr_forecasts.csv', path_lower='/nov2025_ttstr_forecasts.csv', preview_url=NOT_SET, property_groups=NOT_SET, rev='64e54823258fc7a19c0a3', server_modified=datetime.datetime(2026, 3, 31, 16, 32, 8), sharing_info=NOT_SET, size=947382, symlink_info=NOT_SET)

In [20]:
upload(dbx, 'nov2025_TTStr_horizon_metrics.csv', '', '',
            'nov2025_TTStr_horizon_metrics.csv', overwrite=True)

uploaded as b'nov2025_TTStr_horizon_metrics.csv'


FileMetadata(client_modified=datetime.datetime(2026, 3, 31, 16, 34, 58), content_hash='d14ab5d4df7ecacbd29bf4841f6dbd3bed422c254c95c562f383a924e10e56bf', export_info=NOT_SET, file_lock_info=NOT_SET, has_explicit_shared_members=NOT_SET, id='id:oNSmVCFixyQAAAAAAAB85A', is_downloadable=True, media_info=NOT_SET, name='nov2025_TTStr_horizon_metrics.csv', parent_shared_folder_id=NOT_SET, path_display='/nov2025_TTStr_horizon_metrics.csv', path_lower='/nov2025_ttstr_horizon_metrics.csv', preview_url=NOT_SET, property_groups=NOT_SET, rev='64e548cd337317a19c0a3', server_modified=datetime.datetime(2026, 3, 31, 16, 35, 7), sharing_info=NOT_SET, size=1049, symlink_info=NOT_SET)

In [21]:
from tqdm.notebook import tqdm
import pandas as pd


# weather_df = pd.read_csv('https://www.dropbox.com/scl/fi/gmhwwld9z9yychg4r0yuk/weather.csv?rlkey=66c78m90aviamr0x0uu72pfr8&raw=1')
# weather_df.ds = pd.to_datetime(weather_df.ds, errors="coerce")

# df_with_staffing = df.merge(hourly_shifts_by_user_df, on='ds')
# all_variable_df = add_holiday_flags(df_with_staffing, ts_col='ds', include_names=True).merge(weather_df, on='ds')

# Define the start and end dates for November 2025
start_date = pd.Timestamp('2025-11-01 00:00:00')
end_date = pd.Timestamp('2025-11-30 23:00:00')
# end_date = pd.Timestamp('2025-11-01 3:00:00')

basic_output_df = pd.DataFrame()


# Generate a date range for each hour in November 2025
hourly_timestamps = pd.date_range(start=start_date, end=end_date, freq='h')

# Iterate through the timestamps with a progress bar
for timestamp in tqdm(hourly_timestamps, desc="Processing November 2025 hours"):
  target_df = df[df['ds'] < timestamp]
  # print(target_df.tail(5))
  # df_with_holidays = add_holiday_flags(target_df, ts_col='ds', include_names=True)

  # #create a dataframe with the next 24 hours timestamps hourly as column 'ds', with column 'id' jgh
  # future_df = hourly_shifts_by_user_df.reset_index()[hourly_shifts_by_user_df.reset_index()['ds'] > target_df['ds'].max()].head(24)
  # future_df['id'] = 'jgh'
  # future_df = add_holiday_flags(future_df, ts_col='ds', include_names=True)

  # # First, add holiday flags to future_df
  # future_df_with_added_holidays = add_holiday_flags(future_df, ts_col='ds', include_names=True)

  # # Then, select only the columns from future_df_with_added_holidays that are also in df_with_holidays
  # common_columns = [col for col in future_df_with_added_holidays.columns if col in df_with_holidays.columns]
  # future_df_with_holidays = future_df_with_added_holidays[common_columns]

#   future_df_with_staffing = hourly_shifts_by_user_df.reset_index()[hourly_shifts_by_user_df.reset_index()['ds'] > target_df['ds'].max()].head(24)
#   future_df_with_staffing['id'] = 'jgh'

#   future_weather_df = weather_df[weather_df.ds > target_df.ds.max()].head(24)
#   future_weather_df['id']='jgh'


  basic_forecast = pipeline.predict_df(
      df[df.ds<timestamp],
      prediction_length=24,
      #future_df should be future_df_with_staffing merged with future_weather_df on 'ds' and 'id'
    #   future_df = future_df_with_staffing.merge(future_weather_df, on=['ds', 'id']),
      # quantile_levels=[0.2, 0.5, 0.8],
      quantile_levels=[0.5],
      id_column=ID_COL,
      timestamp_column=TS_COL,
      target=TARGETS,
    )
  basic_forecast['cutoff_ts'] = timestamp
  # add the real value from df at each value of ds by merging the dataframes on ds, keeping only the TTStr column from df
  basic_forecast = basic_forecast.merge(df[['ds', 'TTStr']], on='ds', how='left')
  # print(forecast_all_vars_with_future
  # print(forecast_all_vars_with_future[['cutoff_ts', 'ds', 'TTStr', 'predictions']])
  basic_output_df = pd.concat([basic_output_df, basic_forecast[['cutoff_ts', 'ds', 'TTStr', 'predictions']]])

basic_output_df





Processing November 2025 hours:   0%|          | 0/720 [00:00<?, ?it/s]

,cutoff_ts,ds,TTStr,predictions
0,2025-11-01 00:00:00,2025-11-01 00:00:00,90.0,88.871002
1,2025-11-01 00:00:00,2025-11-01 01:00:00,85.0,84.969032
2,2025-11-01 00:00:00,2025-11-01 02:00:00,82.0,83.018532
3,2025-11-01 00:00:00,2025-11-01 03:00:00,81.0,81.468483
4,2025-11-01 00:00:00,2025-11-01 04:00:00,80.0,81.001610
...,...,...,...,...
19,2025-11-30 23:00:00,2025-12-01 18:00:00,130.0,107.591469
20,2025-11-30 23:00:00,2025-12-01 19:00:00,110.0,104.353455
21,2025-11-30 23:00:00,2025-12-01 20:00:00,109.0,101.206520
22,2025-11-30 23:00:00,2025-12-01 21:00:00,104.0,97.416939


In [22]:
forecast = basic_output_df.copy()

#calculate rmse and mape of the forecasts
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
# drop rows with NaN in TTStr
eval_df = forecast.dropna(subset=['TTStr'])
rmse = root_mean_squared_error(eval_df['TTStr'], eval_df['predictions'])
mape = mean_absolute_percentage_error(eval_df['TTStr'], eval_df['predictions'])
print(f"RMSE: {rmse}")
print(f"MAPE: {mape}")

RMSE: 10.11211026490959
MAPE: 0.08615630742029722


In [23]:
#calculate rmse and mape at each forecast horizon (1h ahead, 2h ahead, etc.)
eval_df['horizon'] = (eval_df['ds'] - eval_df['cutoff_ts']).dt.total_seconds() / 3600
horizon_metrics = eval_df.groupby('horizon').apply(lambda x: pd.Series({
    'rmse': root_mean_squared_error(x['TTStr'], x['predictions']),
    'mape': mean_absolute_percentage_error(x['TTStr'], x['predictions'])
}))


horizon_metrics

/tmp/ipykernel_1072612/1665450101.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  horizon_metrics = eval_df.groupby('horizon').apply(lambda x: pd.Series({


,rmse,mape
horizon,,
0.0,4.680061,0.038653
1.0,6.121381,0.051510
2.0,7.206563,0.060722
3.0,7.971327,0.066956
4.0,8.587389,0.071789
5.0,9.089284,0.076765
6.0,9.447164,0.080917
7.0,9.694282,0.083813
8.0,9.957937,0.086882
